# NB2 — Feature Extraction (EAR / MAR / Head Pose)

**Prerequisites:** NB1 must have been run (datasets cached to Drive).

**Steps:**
1. Install packages + import libraries
2. Mount Drive, configure paths, restore datasets to local SSD
3. Download MediaPipe `face_landmarker.task` model (cached to Drive)
4. Define `calc_ear`, `calc_mar`, `get_head_pose` helper functions
5. Explore NTHU-DDD folder structure
6. Extract features from all ~41k NTHU-DDD images → `nthu_features.csv`

> **Time:** First run ~2-3 hours on 41k images.
> Subsequent runs load from Drive cache in seconds.

---

In [1]:
# Install packages (run once per Colab session)
!pip install -q kagglehub mediapipe opencv-python-headless \
    pygame imbalanced-learn tqdm seaborn "protobuf>=5.26.1,<6.0.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.6 MB/s eta 0:00:00


In [2]:
# Common imports – run this cell first
import os, sys, cv2, random, time, shutil, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mediapipe as mp
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score,
    accuracy_score, f1_score, precision_score, recall_score)

np.random.seed(42)
tf.random.set_seed(42)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print('GPU:', gpus[0].name)
else:
    print('NO GPU found - go to Runtime > Change runtime type > T4 GPU')
print('TF:', tf.__version__, '  MP:', mp.__version__)


GPU: /physical_device:GPU:0
TF: 2.19.0   MP: 0.10.32


In [3]:
# Mount Google Drive and configure all paths
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT    = '/content/drive/MyDrive/drowsiness'
MODEL_DIR     = DRIVE_ROOT + '/models'
DATASET_DIR   = DRIVE_ROOT + '/datasets'
LOCAL_DS_ROOT = '/content/datasets'

for d in [MODEL_DIR, DATASET_DIR, LOCAL_DS_ROOT]:
    os.makedirs(d, exist_ok=True)

# Local dataset paths (populated by NB1 download or restored below)
nthu_path     = os.path.join(LOCAL_DS_ROOT, 'nthu_ddd')
mrl_path      = os.path.join(LOCAL_DS_ROOT, 'mrl_eye')
combined_path = os.path.join(LOCAL_DS_ROOT, 'combined')
yawdd_path    = os.path.join(LOCAL_DS_ROOT, 'yawdd')
cew_path      = os.path.join(LOCAL_DS_ROOT, 'cew')

# Model save/load paths
IR_MODEL_PATH     = os.path.join(MODEL_DIR, 'ir_cnn.h5')
RGB_MODEL_PATH    = os.path.join(MODEL_DIR, 'rgb_cnn.h5')
LSTM_MODEL_PATH   = os.path.join(MODEL_DIR, 'lstm.h5')
FUSION_MODEL_PATH = os.path.join(MODEL_DIR, 'fusion.h5')

# Feature CSV
CSV_PATH       = '/content/nthu_features.csv'
CSV_DRIVE_PATH = os.path.join(MODEL_DIR, 'nthu_features.csv')

# MediaPipe FaceLandmarker model
FACE_LANDMARKER_LOCAL = '/content/face_landmarker.task'
FACE_LANDMARKER_DRIVE = os.path.join(MODEL_DIR, 'face_landmarker.task')

EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
print('Paths configured.')
print('  Model dir  :', MODEL_DIR)
print('  Dataset dir:', LOCAL_DS_ROOT)


Mounted at /content/drive
Paths configured.
  Model dir  : /content/drive/MyDrive/drowsiness/models
  Dataset dir: /content/datasets


In [4]:
# Restore datasets from Drive to local SSD (needed on new session)
# Skip any dataset that is already extracted.
import tarfile

DATASET_NAMES = {
    'nthu': 'nthu_ddd', 'mrl': 'mrl_eye', 'combined': 'combined',
    'yawdd': 'yawdd',   'cew': 'cew',
}
for key, folder in DATASET_NAMES.items():
    local_path   = os.path.join(LOCAL_DS_ROOT, folder)
    archive_path = os.path.join(DATASET_DIR, folder + '.tar.gz')
    if os.path.isdir(local_path) and any(os.scandir(local_path)):
        print('[OK    ]', key, '- already on local SSD')
    elif os.path.isfile(archive_path):
        print('[UNPACK]', key, '...')
        os.makedirs(local_path, exist_ok=True)
        with tarfile.open(archive_path, 'r:gz') as tar:
            tar.extractall(local_path)
        print('        ->', local_path)
    else:
        print('[MISSING]', key, '- run NB1_Setup_and_Data.ipynb first!')


[UNPACK] nthu ...


/tmp/ipykernel_1981/2337526951.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(local_path)


        -> /content/datasets/nthu_ddd
[UNPACK] mrl ...
        -> /content/datasets/mrl_eye
[UNPACK] combined ...
        -> /content/datasets/combined
[UNPACK] yawdd ...
        -> /content/datasets/yawdd
[UNPACK] cew ...
        -> /content/datasets/cew


---
## Section 3 â€” Feature Extraction (EAR / MAR / Head Pose)
Define MediaPipe-based functions used throughout the pipeline.

In [5]:
# ── Download face_landmarker.task model (cached to Drive) ─────────────────────
FACE_LANDMARKER_LOCAL = '/content/face_landmarker.task'
FACE_LANDMARKER_DRIVE = os.path.join(MODEL_DIR, 'face_landmarker.task')
_FL_URL = ('https://storage.googleapis.com/mediapipe-models/'
           'face_landmarker/face_landmarker/float16/1/face_landmarker.task')

if os.path.isfile(FACE_LANDMARKER_LOCAL):
    print('[LOCAL ] face_landmarker.task already on SSD')
elif os.path.isfile(FACE_LANDMARKER_DRIVE):
    shutil.copy(FACE_LANDMARKER_DRIVE, FACE_LANDMARKER_LOCAL)
    print('[CACHED] face_landmarker.task from Drive')
else:
    print('Downloading face_landmarker.task …')
    urllib.request.urlretrieve(_FL_URL, FACE_LANDMARKER_LOCAL)
    shutil.copy(FACE_LANDMARKER_LOCAL, FACE_LANDMARKER_DRIVE)
    print('[SAVED ] face_landmarker.task → Drive')

# ── Tasks API aliases (MediaPipe 0.10.13+) ────────────────────────────────────
_BaseOptions        = mp.tasks.BaseOptions
_FaceLandmarker     = mp.tasks.vision.FaceLandmarker
_FaceLandmarkerOpts = mp.tasks.vision.FaceLandmarkerOptions
_RunningMode        = mp.tasks.vision.RunningMode

def make_face_landmarker():
    """Return a FaceLandmarker context-manager (IMAGE mode, 1 face)."""
    opts = _FaceLandmarkerOpts(
        base_options=_BaseOptions(model_asset_path=FACE_LANDMARKER_LOCAL),
        running_mode=_RunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return _FaceLandmarker.create_from_options(opts)

# ── MediaPipe landmark indices (same for old FaceMesh & new FaceLandmarker) ───
LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33,  160, 158, 133, 153, 144]
MOUTH     = [61,  39,  37,  0,  267, 269, 291, 405, 314, 17, 84, 181]
POSE_IDS  = [1, 152, 33, 263, 61, 291]
FACE_3D   = np.array([
    [0.,0.,0.], [0.,-330.,-65.],
    [-225.,170.,-135.], [225.,170.,-135.],
    [-150.,-150.,-125.], [150.,-150.,-125.]
], dtype=np.float64)
EAR_THRESHOLD = 0.25
MAR_THRESHOLD = 0.60

def calc_ear(lm, ids, w, h):
    pts = [np.array([lm[i].x*w, lm[i].y*h]) for i in ids]
    A = np.linalg.norm(pts[1]-pts[5])
    B = np.linalg.norm(pts[2]-pts[4])
    C = np.linalg.norm(pts[0]-pts[3])
    return (A+B) / (2*C + 1e-7)

def calc_mar(lm, ids, w, h):
    pts = [np.array([lm[i].x*w, lm[i].y*h]) for i in ids]
    A=np.linalg.norm(pts[1]-pts[9]); B=np.linalg.norm(pts[2]-pts[10])
    C=np.linalg.norm(pts[3]-pts[11]); D=np.linalg.norm(pts[0]-pts[6])
    return (A+B+C) / (2*D + 1e-7)

def get_head_pose(lm, w, h):
    img_pts = np.array([[lm[i].x*w, lm[i].y*h] for i in POSE_IDS], dtype=np.float64)
    cam = np.array([[w,0,w/2],[0,w,h/2],[0,0,1]], dtype=np.float64)
    ok, rvec, _ = cv2.solvePnP(FACE_3D, img_pts, cam, np.zeros((4,1)),
                                flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok: return 0., 0., 0.
    rmat, _ = cv2.Rodrigues(rvec)
    sy = np.sqrt(rmat[0,0]**2 + rmat[1,0]**2)
    pitch = float(np.degrees(np.arctan2(-rmat[2,0], sy)))
    yaw   = float(np.degrees(np.arctan2(rmat[1,0], rmat[0,0])))
    roll  = float(np.degrees(np.arctan2(rmat[2,1], rmat[2,2])))
    return pitch, yaw, roll

print("Feature extraction functions defined.")
print("Feature vector order: [EAR, MAR, pitch, yaw, IR_score]")


[SAVED ] face_landmarker.task → Drive
Feature extraction functions defined.
Feature vector order: [EAR, MAR, pitch, yaw, IR_score]


In [6]:
# ── NTHU-DDD folder explorer — run once to find exact video paths ─────────────
print(f"Scanning: {nthu_path}\n")

for root, dirs, files in os.walk(nthu_path):
    level = root.replace(nthu_path, '').count(os.sep)
    if level > 4:           # don't go too deep
        dirs[:] = []
        continue
    indent = '  ' * level
    rel = os.path.relpath(root, nthu_path)
    print(f"{indent}{rel}/  ({len(files)} files)")
    if files:
        # show extension breakdown + first 2 filenames
        ext_count = Counter(Path(f).suffix.lower() for f in files)
        print(f"{indent}  extensions: {dict(ext_count)}")
        for f in files[:2]:
            print(f"{indent}  ↳ {f}")


Scanning: /content/datasets/nthu_ddd

./  (0 files)
  Driver Drowsiness Dataset (DDD)/  (0 files)
    Driver Drowsiness Dataset (DDD)/Non Drowsy/  (19445 files)
      extensions: {'.png': 19445}
      ↳ d0698.png
      ↳ e0548.png
    Driver Drowsiness Dataset (DDD)/Drowsy/  (22348 files)
      extensions: {'.png': 22348}
      ↳ W0742.png
      ↳ T0384.png


In [7]:
# Extract EAR / MAR / head-pose features from NTHU-DDD frame images
# Dataset structure (confirmed):
#   Driver Drowsiness Dataset (DDD)/Drowsy/     22,348 .png
#   Driver Drowsiness Dataset (DDD)/Non Drowsy/ 19,445 .png
CSV_PATH       = '/content/nthu_features.csv'
CSV_DRIVE_PATH = os.path.join(MODEL_DIR, 'nthu_features.csv')

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def label_from_path(path_str):
    """
    Infer label from parent folder name.
    CRITICAL: not-drowsy variants MUST be checked before 'drowsy' because
    'non drowsy' and 'not drowsy' both contain the substring 'drowsy'.
    """
    # Use the immediate parent folder name for precision
    folder = Path(path_str).parent.name.lower()
    NOT_DROWSY = ('non drowsy', 'not drowsy', 'notdrowsy', 'non_drowsy',
                  'not_drowsy', 'nondrowsy', 'normal', 'alert')
    SLIGHTLY   = ('slightly drowsy', 'slightly', 'slight drowsy', 'slight')

    if any(folder == x or folder.startswith(x) for x in NOT_DROWSY): return 0
    if any(x in folder for x in SLIGHTLY):                            return 1
    if 'drowsy' in folder:                                            return 2
    return None

# ── 1. Drive cache ─────────────────────────────────────────────────────────
if os.path.isfile(CSV_DRIVE_PATH):
    shutil.copy(CSV_DRIVE_PATH, CSV_PATH)
    df_feats = pd.read_csv(CSV_PATH)
    print(f"[CACHED] Loaded {len(df_feats)} feature rows from Drive")

# ── 2. Extract from NTHU-DDD images ───────────────────────────────────────
elif os.path.isdir(nthu_path):
    all_imgs = [p for p in Path(nthu_path).rglob('*')
                if p.suffix.lower() in IMG_EXTS]
    print(f"Found {len(all_imgs)} images")

    labeled = [(str(p), label_from_path(str(p))) for p in all_imgs]
    labeled = [(fp, lbl) for fp, lbl in labeled if lbl is not None]
    unlabeled = len(all_imgs) - len(labeled)
    print(f"Labeled  : {len(labeled)}  "
          f"alert={sum(l==0 for _,l in labeled)}  "
          f"slight={sum(l==1 for _,l in labeled)}  "
          f"drowsy={sum(l==2 for _,l in labeled)}")
    if unlabeled:
        print(f"Unlabeled: {unlabeled} (folder name not recognised)")

    records = []
    skip = 0
    with make_face_landmarker() as landmarker:
        for i, (fp, label) in enumerate(labeled):
            img = cv2.imread(fp)
            if img is None:
                skip += 1
                continue
            rgb    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect(mp_img)
            if result.face_landmarks:
                lm = result.face_landmarks[0]
                h, w = img.shape[:2]
                ear = (calc_ear(lm, LEFT_EYE, w, h) +
                       calc_ear(lm, RIGHT_EYE, w, h)) / 2
                mar = calc_mar(lm, MOUTH, w, h)
                p_val, y_val, _ = get_head_pose(lm, w, h)
                records.append({'EAR': ear, 'MAR': mar,
                                'pitch': p_val, 'yaw': y_val,
                                'IR_score': 0.0, 'label': label})
            if (i + 1) % 2000 == 0:
                print(f"  {i+1}/{len(labeled)}  faces detected: {len(records)}")

    print(f"Done — faces detected: {len(records)} / {len(labeled)}  "
          f"(unreadable: {skip})")
    df_feats = pd.DataFrame(records)
    df_feats.to_csv(CSV_PATH, index=False)
    shutil.copy(CSV_PATH, CSV_DRIVE_PATH)
    print(f"Saved + backed up to Drive → {CSV_DRIVE_PATH}")

# ── 3. Synthetic fallback ──────────────────────────────────────────────────
else:
    print("[WARNING] nthu_path not found — using synthetic data.")
    np.random.seed(42); N = 5000
    df_feats = pd.DataFrame({
        'EAR': np.random.uniform(0.15,0.40,N), 'MAR': np.random.uniform(0.20,0.80,N),
        'pitch': np.random.uniform(-30,30,N),   'yaw': np.random.uniform(-20,20,N),
        'IR_score': np.zeros(N),
        'label': np.random.choice([0,1,2], N, p=[0.5,0.25,0.25]),
    })
    df_feats.to_csv(CSV_PATH, index=False)

print(f"\nFeature CSV ready: {len(df_feats)} rows")
print(df_feats['label'].value_counts().rename({0:'alert', 1:'slight', 2:'drowsy'}))


Found 41793 images
Labeled  : 41793  alert=19445  slight=0  drowsy=22348
  2000/41793  faces detected: 1999
  4000/41793  faces detected: 3997
  6000/41793  faces detected: 5995
  8000/41793  faces detected: 7994
  10000/41793  faces detected: 9992
  12000/41793  faces detected: 11992
  14000/41793  faces detected: 13992
  16000/41793  faces detected: 15992
  18000/41793  faces detected: 17992
  20000/41793  faces detected: 19991
  22000/41793  faces detected: 21990
  24000/41793  faces detected: 23988
  26000/41793  faces detected: 25988
  28000/41793  faces detected: 27988
  30000/41793  faces detected: 29987
  32000/41793  faces detected: 31984
  34000/41793  faces detected: 33981
  36000/41793  faces detected: 35979
  38000/41793  faces detected: 37978
  40000/41793  faces detected: 39977
Done — faces detected: 41769 / 41793  (unreadable: 0)
Saved + backed up to Drive → /content/drive/MyDrive/drowsiness/models/nthu_features.csv

Feature CSV ready: 41769 rows
label
drowsy    22332
a